In [9]:
import os
import json
import mlflow
from dotenv import load_dotenv
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml.entities import (
    Workspace,
    Model,
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Environment,
    CodeConfiguration
)
from azure.ai.ml.constants import AssetTypes
from azure.mgmt.resource import ResourceManagementClient

# ─────────────────────────────────────────────
# CONFIGURACIÓN GLOBAL
# ─────────────────────────────────────────────

load_dotenv()
subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID")

resource_group  = "rg-credit-risk-ml"
workspace_name  = "ws-credit-risk-ml"
location        = "eastus"

credential = InteractiveBrowserCredential(
    tenant_id="adcc6d74-c336-483f-b3e0-508ef38f11d8"
)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

print(f"✅ Subscription: {subscription_id[:8]}...")
print(f"✅ Workspace: {ml_client.workspace_name}")
print(f"✅ Listo para operar")

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


✅ Subscription: 7a434baa...
✅ Workspace: ws-credit-risk-ml
✅ Listo para operar


In [7]:
# ─────────────────────────────────────────────
# IMPORTACIONES
# ─────────────────────────────────────────────

import os
# Módulo estándar de Python para interactuar con
# variables de entorno del sistema operativo

from dotenv import load_dotenv
# Lee el archivo .env y carga sus variables como
# variables de entorno accesibles con os.environ

from azure.ai.ml import MLClient
# Cliente principal del SDK de Azure ML —
# es el punto de entrada para todas las operaciones:
# crear workspaces, registrar modelos, crear endpoints, etc.

from azure.identity import InteractiveBrowserCredential
# Maneja la autenticación abriendo el navegador
# para que inicies sesión con tu cuenta Microsoft.
# Genera un token OAuth que autoriza las llamadas a Azure.

from azure.ai.ml.entities import Workspace
# Clase que representa un Azure ML Workspace —
# es el contenedor principal de todos los recursos ML:
# modelos, experimentos, compute, endpoints, etc.

from azure.mgmt.resource import ResourceManagementClient
# Cliente para gestionar recursos base de Azure
# como Resource Groups — contenedores lógicos de recursos

# ─────────────────────────────────────────────
# CARGAR CREDENCIALES
# ─────────────────────────────────────────────

load_dotenv()
# Busca el archivo .env en el directorio actual
# y carga sus variables en el entorno del proceso

subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID")
# Lee la variable AZURE_SUBSCRIPTION_ID del entorno
# El ID identifica tu suscripción de Azure — sin él
# Azure no sabe a qué cuenta asociar los recursos

print(f"Subscription ID cargado: {subscription_id[:8]}...")
# Muestra solo los primeros 8 caracteres por seguridad

# ─────────────────────────────────────────────
# PARÁMETROS
# ─────────────────────────────────────────────

resource_group = "rg-credit-risk-ml"
# Contenedor lógico de Azure que agrupa recursos relacionados.
# Convención de nombre: prefijo "rg-" + nombre del proyecto.
# Facilita gestión de costos y permisos por proyecto.

workspace_name  = "ws-credit-risk-ml"
# Nombre del Azure ML Workspace.
# Convención: prefijo "ws-" + nombre del proyecto.

location = "eastus"
# Región de Azure donde se crearán los recursos.
# eastus (Virginia, EE.UU.) es la más económica y
# con mayor disponibilidad de servicios de ML.

# ─────────────────────────────────────────────
# AUTENTICACIÓN
# ─────────────────────────────────────────────

credential = InteractiveBrowserCredential(
    tenant_id="adcc6d74-c336-483f-b3e0-508ef38f11d8"
)
# Instancia el manejador de autenticación especificando
# el tenant del "Directorio predeterminado" de Azure.
# Sin esto, el navegador autenticaría en el tenant
# personal de Microsoft (incorrecto para esta suscripción).

# ─────────────────────────────────────────────
# CREAR RESOURCE GROUP
# ─────────────────────────────────────────────

resource_client = ResourceManagementClient(
    credential=credential,
    subscription_id=subscription_id
)
# ResourceManagementClient gestiona recursos base de Azure.
# Necesario para crear el Resource Group antes del Workspace.

rg_result = resource_client.resource_groups.create_or_update(
    resource_group,
    {"location": location}
)
# create_or_update: crea el grupo si no existe,
# o lo deja intacto si ya existe — operación segura
# para ejecutar múltiples veces sin efectos secundarios.

print(f"Resource Group '{rg_result.name}' listo en {rg_result.location}")

# ─────────────────────────────────────────────
# CREAR MLCLIENT
# ─────────────────────────────────────────────

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group
)
# MLClient es el objeto central del SDK.
# Recibe las credenciales y la ubicación de los recursos.
# A través de él accedemos a:
#   ml_client.workspaces       → gestión de workspaces
#   ml_client.models           → registro de modelos
#   ml_client.online_endpoints → despliegue de APIs
#   ml_client.jobs             → pipelines y entrenamientos

# ─────────────────────────────────────────────
# CREAR WORKSPACE
# ─────────────────────────────────────────────

ws = Workspace(
    name=workspace_name,
    location=location,
    display_name="Credit Risk Scoring ML",
    # Nombre legible que aparece en el portal Azure
    description="Workspace para modelo de scoring de riesgo crediticio"
    # Documentación del propósito del workspace
)

ws = ml_client.workspaces.begin_create(ws).result()
# begin_create() inicia la creación de forma asíncrona
# .result() espera a que termine (puede tardar 2-3 minutos)
# Azure crea automáticamente recursos asociados:
#   - Storage Account  (almacenamiento de artefactos)
#   - Key Vault        (gestión de secretos)
#   - Application Insights (monitoreo)
#   - Container Registry   (imágenes Docker para endpoints)

print(f"Workspace creado: {ws.name} en {ws.location}")

Subscription ID cargado: 7a434baa...


C:\Users\Marin\.conda\envs\credit-risk\Lib\site-packages\msal\oauth2cli\oauth2.py:408: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Resource Group 'rg-credit-risk-ml' listo en eastus


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Workspace creado: ws-credit-risk-ml en eastus


In [8]:
# Listar todos los modelos registrados en MLflow local
modelos = client.search_registered_models()

for modelo in modelos:
    print(f"Nombre: {modelo.name}")
    print(f"Aliases: {modelo.aliases}")
    print("─" * 40)

NameError: name 'client' is not defined

In [ ]:
# ─────────────────────────────────────────────
# LOCALIZAR EL MODELO EN MLFLOW LOCAL
# ─────────────────────────────────────────────

model_version = client.get_model_version_by_alias(
    name="lightgbm_scoring",
    # Nombre correcto del modelo registrado en MLflow
    alias="production"
    # Alias que apunta a la versión productiva (versión 3)
)

model_uri = model_version.source
# URI local donde están los artefactos del modelo

print(f"Modelo encontrado en: {model_uri}")
print(f"Versión: {model_version.version}")
print(f"Run ID: {model_version.run_id}")

In [ ]:
# ─────────────────────────────────────────────
# RESOLVER RUTA FÍSICA DEL MODELO
# ─────────────────────────────────────────────

local_model_path = mlflow.artifacts.download_artifacts(
    artifact_uri=model_uri
)
# download_artifacts resuelve el URI interno de MLflow
# y devuelve la ruta física real en disco donde están
# los artefactos: modelo, parámetros, requirements, etc.

print(f"Ruta física del modelo: {local_model_path}")

In [ ]:
# ─────────────────────────────────────────────
# RECONECTAR MLCLIENT CON WORKSPACE
# ─────────────────────────────────────────────

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
    # workspace_name es obligatorio para operaciones
    # sobre modelos, endpoints y jobs — sin él el
    # cliente no sabe en qué workspace operar
)

print(f"MLClient conectado a: {ml_client.workspace_name}")

In [ ]:
# ─────────────────────────────────────────────
# REGISTRAR MODELO EN AZURE ML
# ─────────────────────────────────────────────

model = Model(
    path=local_model_path,
    # Ruta local donde están los artefactos del modelo

    name="lightgbm-credit-risk",
    # Nombre del modelo en Azure ML Model Registry
    # Convención: kebab-case (guiones, no guiones bajos)

    description="Modelo LightGBM para scoring de riesgo crediticio. "
                "AUC=0.768 en conjunto de prueba. "
                "Entrenado con dataset Home Credit Default Risk.",
    # Documentación visible en el portal Azure

    type=AssetTypes.MLFLOW_MODEL,
    # Indica que es un modelo en formato MLflow —
    # Azure ML lo reconoce y puede servir directamente
    # sin necesidad de escribir scoring script manual

    tags={
        "framework": "lightgbm",
        "task": "credit_risk_scoring",
        "auc": "0.768",
        "dataset": "home_credit_default_risk",
        "mlflow_version": model_version.version,
        "mlflow_alias": "production"
    }
    # Metadatos clave-valor para búsqueda y filtrado
    # en el portal Azure — equivalente a los tags de MLflow
)

registered_model = ml_client.models.create_or_update(model)
# Sube los artefactos al Storage Account del Workspace
# y registra el modelo en Azure ML Model Registry

print(f"Modelo registrado: {registered_model.name}")
print(f"Versión en Azure ML: {registered_model.version}")
print(f"ID: {registered_model.id}")

In [14]:
# ─────────────────────────────────────────────
# CREAR ENDPOINT
# ─────────────────────────────────────────────

endpoint_name = "cr-scoring-marin-v2"
# Nombre del endpoint — debe ser único en Azure
# Se convertirá en parte de la URL de la API

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Endpoint para scoring de riesgo crediticio en tiempo real",
    auth_mode="key"
    # auth_mode="key" significa que las llamadas a la API
    # requieren una API key — protege el endpoint de acceso no autorizado
)

poller = ml_client.online_endpoints.begin_create_or_update(endpoint)
# Inicia la creación del endpoint de forma asíncrona
# NO usamos .result() para no bloquear el kernel

print("⏳ Endpoint en creación...")
print("Verifica el estado en Azure ML Studio → Puntos de conexión")

⏳ Endpoint en creación...
Verifica el estado en Azure ML Studio → Puntos de conexión


In [15]:
# ─────────────────────────────────────────────
# CREAR DEPLOYMENT
# ─────────────────────────────────────────────

deployment = ManagedOnlineDeployment(
    name="blue",
    # Nombre de la implementación dentro del endpoint
    # Convención: "blue" para producción, "green" para staging
    # Permite hacer blue/green deployments sin downtime

endpoint_name="cr-scoring-marin-v2",    
    # Endpoint al que pertenece este deployment

    model=f"azureml:lightgbm-credit-risk:1",
    # Referencia al modelo registrado en Azure ML
    # formato: azureml:nombre:version

    code_configuration=CodeConfiguration(
    code="../src/endpoint",
    scoring_script="score.py"
),

environment=Environment(
    name="credit-risk-env",
    description="Entorno para modelo de scoring crediticio",
    conda_file="../src/endpoint/environment.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
),

    instance_type="Standard_DS2_v2",
    # Tipo de VM para el endpoint
    # Standard_DS2_v2: 2 vCPUs, 7GB RAM — el más económico
    # disponible para Managed Online Endpoints

    instance_count=1
    # Número de instancias — 1 es suficiente para pruebas
    # En producción real se usarían 2+ para alta disponibilidad
)

poller = ml_client.online_deployments.begin_create_or_update(deployment)
print("⏳ Deployment en proceso — verifica en Azure ML Studio → Puntos de conexión")
print("Este proceso tarda 10-15 minutos")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint cr-scoring-marin-v2 exists


HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"This endpoint has not been created successfully or is in deleting provisioning state. Please recreate endpoint and then try again to create deployment.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-39e52578916ec89868c4aa676b8328f8-a5faf647472c8c49-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"This endpoint has not been created successfully or is in deleting provisioning state. Please recreate endpoint and then try again to create deployment.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-39e52